# PyMuPDF

In [4]:
import pymupdf

In [5]:
filename = "input_dict_files/rodegem_rundi1970_o.pdf"

rundi_dict_pdf = pymupdf.open(filename)  # or pymupdf.Document(filename)

In [6]:
rundi_sample_page = rundi_dict_pdf.load_page(27)

In [7]:
rundi_sample_page.get_text().split('\n')

['A',
 'a, (inv.), interjection de négation, de dénéga\xad',
 'tion : non. Peut être prononcée à bouche entrou\xad',
 'verte ou à bouche fermée. Dans ce dernier cas, ',
 'elle est souvent redoublée en a a, le second ',
 'étant prononcé légèrement plus haut.',
 'umwaba, imy-, 3|4, surface de terre arable dans ',
 'le marais. Mw-irima ry’imyaba, durant la cul\xad',
 'ture des marais.',
 'akâba, utw-, 12|13, lopin de terre arable. Drain ',
 "dans les champs de marais. Akâba n'umugazo ",
 'barekérwa mu busïze bw’îmfûnzo, dans le ter\xad',
 'rain défriché, parmi les papyrus on dégage un ',
 'drain appelé akâba.',
 'kwâba, -âvye, crier vers; avoir recours à, ',
 'recourir à. Hàgira ikibâye gishikiye umuryango ',
 'bàca bâgenda kwâba umupfùmu, si un malheur ',
 'frappait la famille, ils allaient aussitôt consulter ',
 'le devin.',
 'kwabàba, -vye, (-ab--), tendre la main pour ',
 'prendre en se dressant sur la pointe des pieds. ',
 'Il Commencer à ramper (enfant). Syn. kwavüra. ',
 '|( ',
 'E

In [12]:
import re

def insert_start_end_tokens(text: str) -> str:
    """
    Inserts '<start>' before each dictionary entry.

    Assumes each entry starts at a new line and begins with 2
    comma-separated fields, with the second .
    """

    # Step 1: insert <start>
    start_pattern = re.compile(
        r'^((?:[^,\n]+,\s*){1,2}\d+(?:\|\d+)?)',
        re.MULTILINE
    )
    text = start_pattern.sub(r'<start>\1', text)

    # # Step 2: for each <start>, find first '.' and insert <end>
    # def add_end_token(match):
    #     segment = match.group(0)
    #     dot_index = segment.find('.')
    #     if dot_index != -1:
    #         return segment[:dot_index + 1] + '<end>' + segment[dot_index + 1:]
    #     return segment  # no full stop found

    # Match from <start> up to next <start> or end of text
    entry_pattern = re.compile(r'<start>.*?(?=<start>|$)', re.DOTALL)
    # text = entry_pattern.sub(add_end_token, text)

    return text

In [16]:
# result = insert_start_end_tokens(rundi_sample_page.get_text())
# insert_start_end_tokens(rundi_sample_page.get_text())

"A\na, (inv.), interjection de négation, de dénéga\xad\ntion : non. Peut être prononcée à bouche entrou\xad\nverte ou à bouche fermée. Dans ce dernier cas, \nelle est souvent redoublée en a a, le second \nétant prononcé légèrement plus haut.\n<start>umwaba, imy-, 3|4, surface de terre arable dans \nle marais. Mw-irima ry’imyaba, durant la cul\xad\nture des marais.\n<start>akâba, utw-, 12|13, lopin de terre arable. Drain \ndans les champs de marais. Akâba n'umugazo \nbarekérwa mu busïze bw’îmfûnzo, dans le ter\xad\nrain défriché, parmi les papyrus on dégage un \ndrain appelé akâba.\nkwâba, -âvye, crier vers; avoir recours à, \nrecourir à. Hàgira ikibâye gishikiye umuryango \nbàca bâgenda kwâba umupfùmu, si un malheur \nfrappait la famille, ils allaient aussitôt consulter \nle devin.\nkwabàba, -vye, (-ab--), tendre la main pour \nprendre en se dressant sur la pointe des pieds. \nIl Commencer à ramper (enfant). Syn. kwavüra. \n|( \nEtre presque \négal, approcher \nde prés. \nUwàbâba umukù

In [17]:
# for entry in result.split('<start>'):
#     print(entry)
#     print('\n')

A
a, (inv.), interjection de négation, de dénéga­
tion : non. Peut être prononcée à bouche entrou­
verte ou à bouche fermée. Dans ce dernier cas, 
elle est souvent redoublée en a a, le second 
étant prononcé légèrement plus haut.



umwaba, imy-, 3|4, surface de terre arable dans 
le marais. Mw-irima ry’imyaba, durant la cul­
ture des marais.



akâba, utw-, 12|13, lopin de terre arable. Drain 
dans les champs de marais. Akâba n'umugazo 
barekérwa mu busïze bw’îmfûnzo, dans le ter­
rain défriché, parmi les papyrus on dégage un 
drain appelé akâba.
kwâba, -âvye, crier vers; avoir recours à, 
recourir à. Hàgira ikibâye gishikiye umuryango 
bàca bâgenda kwâba umupfùmu, si un malheur 
frappait la famille, ils allaient aussitôt consulter 
le devin.
kwabàba, -vye, (-ab--), tendre la main pour 
prendre en se dressant sur la pointe des pieds. 
Il Commencer à ramper (enfant). Syn. kwavüra. 
|( 
Etre presque 
égal, approcher 
de prés. 
Uwàbâba umukùru, vice-président, lieutenant. 
Aramwabâba mu 

# Tesseract

In [1]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageOps, ImageFilter
import time

In [2]:
def ocr_pdf_pipeline(pdf_path, image_dpi=300, languages ="fra+latn", ocr_config = '--oem 3 --psm 3' ):
    """
    Extracts text page-by-page to keep memory usage low.
    """
    full_text = []

    # convert_from_path returns a list of PIL images. 
    # To be truly memory efficient with large PDFs, we process them one by one.
    try:
        # We use a generator-like approach or process in small chunks
        # thread_count can speed up conversion, but increases CPU/RAM usage
        images = convert_from_path(pdf_path, dpi=image_dpi) 

        for i, image in enumerate(images):
            print(f"Processing page {i + 1}...")
            
            # 1. Perform OCR on the single page image
            
            # Specify languages: e.g., 'fra+yor' or 'fra+latn'
            # 'fra' handles the French vocabulary/grammar
            # 'latn' handles Latin characters/accents not specific to French

            # --- CROP LOGIC ---
            width, height = image.size
            
            crop_box = (
                width * 0.05,  # 5% from left
                height * 0.05, # 5% from top
                width * 0.95,  # 95% across (5% from right)
                height * 0.95  # 95% down (5% from bottom)
            )
            
            cropped_image = image.crop(crop_box)
           
            # if you wanna see the images
            cropped_image.save(f'sample_page_{i}.jpg')

            # Execute OCR
            text = pytesseract.image_to_string(cropped_image, lang=languages, config=ocr_config)
        
            # 2. Append text to our list
            full_text.append(text)
            
            # 3. Explicitly clear the image from memory
            cropped_image.close()
            image.close()     
            break
        return "\n\n".join(full_text)

    except Exception as e:
        return f"An error occurred: {e}"

In [33]:
# import os
# os.environ['TESSDATA_PREFIX'] = '/Users/harshamarsha/Documents/CourseWork/Bantu_HiWi/digitizing/tessdata_best/'

In [21]:
filename = "input_dict_files/rundi_sample.pdf"

start_time = time.perf_counter()
extracted_content = ocr_pdf_pipeline(
    
    filename, 
                                     
    image_dpi=900, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = '--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0'
)

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time):.2f} seconds')

Processing page 1...
Time taken to OCR: 18.21 seconds


In [22]:
print(extracted_content)

a, (inv.), interjection de négation, de dénéga-
tion: non. Peut être prononcée à bouche entrou-
verte ou à bouche fermée. Dans ce dernier cas,
eke est souvent redoublée en a a, le second
étant prononcé légèrement plus haut.

umwaba, imy-, 34, surface de terre arable dans
le marais, Mw-irima ryimyvaba, durant ta cul-
fure des maårais.

akāba, utw-, 12|13, lopin de terre arable. Drain
dans les champs de marais. Akaba n'umugazo
barekérwa mu busříze bw'îmfůnzo, dans le ter-
rain défriché, parmi les papyrus on dégage un
drain appelé akāba.

kwâba, -âvye, crier vers; avoir recours ğ,
recourir à. Hšgira ikibâye gishikíye umuryango
băca bågenda kwâôba umupfůmu, si un malheur
frappait la famille, ils allaient aussitôt consulier
le devin.

kwabāäba, -vye, (-ab--), tendre la main pour
prendre en se dressant sur la pointe des pieds.
li Commencer à ramper (enfant). Syn. kwavūra.
| Etre presque égal, approcher de près.
Uwăbāba umukůru, vice-président, lieutenant.
Aramwabāba mu kuvůka, is ont presque

# Tesseract + Line segmentation

In [1]:
import fitz  # PyMuPDF
import time
import os
import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm
import gc # Garbage Collector


# Set Tesseract to use multiple threads internally
os.environ["OMP_THREAD_LIMIT"] = "8"

In [17]:
# def save_entry_to_file(entry_str, file_path="dictionary_output.txt"):
#     """Appends a single entry to the text file immediately."""
#     with open(file_path, "a", encoding="utf-8") as f:
#         f.write(entry_str + "\n\n")
        

# def find_footer_separator_cv2(pil_img):
#     # 1. Convert to OpenCV grayscale
#     img = np.array(pil_img.convert('L'))
#     h, w = img.shape
    
#     # 2. Focus on the bottom 30% and binarize
#     # We use a higher threshold (220) to catch very faint/thin lines
#     start_y = int(h * 0.50)
#     roi = img[start_y:, :]
#     _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
#     # 3. Create a horizontal "needle" kernel
#     # This kernel is 1 pixel tall but 50 pixels wide.
#     # It acts like a comb that only lets horizontal structures through.
#     horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    
#     # 4. Apply Morphology (Erosion followed by Dilation)
#     # This deletes everything that isn't at least 50 pixels wide horizontally.
#     detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
#     # 5. Find the Y-coordinate of the remaining pixels
#     # We look for the row with the most white pixels in the 'detected_lines' mask
#     row_sums = np.sum(detected_lines, axis=1)
    
#     if np.max(row_sums) > 0:
#         # Find the index of the row with the maximum "line-ness"
#         line_relative_y = np.argmax(row_sums)
#         global_y = start_y + line_relative_y
        
#         # Buffer: return 15 pixels above the line to ensure it's fully gone
#         return global_y - 15
        
#     return h

    
# def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn", ocr_config='--oem 3 --psm 3'):
    
#     output_file = "ocr_results.txt"
#     # Clear file if it exists from a previous run
#     if os.path.exists(output_file):
#         os.remove(output_file)
    
#     # Get page count without loading images
#     info = pdfinfo_from_path(pdf_path)
#     max_pages = info["Pages"]
    
#     for page_num in tqdm(range(1, max_pages + 1), desc="OCR Progress", unit="page"): 
        
#         print(f"Processing page {page_num}/{max_pages}...")
        
#         images = convert_from_path(pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num)
#         if not images:
#             continue
            
#         image = images[0]

#         # 1. Detect and Crop Footer
#         footer_y = find_footer_separator_cv2(image)
#         if footer_y < image.size[1]:
#             image = image.crop((0, 0, image.size[0], footer_y))   
                
#         width, height = image.size
#         left_m, right_m = width * 0.05, width * 0.95
#         top_m, mid = height * 0.05, width / 2
        
#         columns = [
#             ("Left", image.crop((left_m, top_m, mid, height))),
#             ("Right", image.crop((mid, top_m, right_m, height)))
#         ]

#         for col_name, col_img in columns:
#             # debug image
#             col_img.save(f'p{page_num}_{col_name}.png')
            
#             data = pytesseract.image_to_data(col_img, lang=languages, config=ocr_config, output_type=Output.DICT)
            
#             # Step 1: Reconstruct lines
#             lines = []
#             n_words = len(data['text'])
#             current_line_text, current_line_x = [], None

#             for j in range(n_words):
#                 if int(data['conf'][j]) > 0:
#                     if int(data['word_num'][j]) == 1:
#                         if current_line_text:
#                             lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})
#                         current_line_x = data['left'][j]
#                         current_line_text = [data['text'][j]]
#                     else:
#                         current_line_text.append(data['text'][j])
            
#             if current_line_text:
#                 lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})

#             if not lines:
#                 continue

#             # Step 2: Thresholding
#             median_x = np.median([l['x'] for l in lines])
#             indent_threshold = 40 # Adjust to 90 if 40 is too sensitive for 600dpi

#             # Step 3: Group and Write to File
#             current_entry = []
#             for line in lines:
#                 is_indent = line['x'] > (median_x + indent_threshold)

#                 if is_indent:
#                     if current_entry:
#                         # --- SAVE POSITION 1: End of an entry detected by a new indent ---
#                         entry_str = " ".join(current_entry).strip()
#                         save_entry_to_file(entry_str, output_file)
                    
#                     current_entry = [line['text']]
#                 else:
#                     if current_entry:
#                         current_entry.append(line['text'])
#                     else:
#                         current_entry = [line['text']]

#             if current_entry:
#                 # --- SAVE POSITION 2: Final entry of the column ---
#                 entry_str = " ".join(current_entry).strip()
#                 save_entry_to_file(entry_str, output_file)

#         # Cleanup Memory
#         for _, c_img in columns:
#             c_img.close()
#         image.close()
#         del images 
#         if page_num == 4: break
#     print(f"Finished! All entries saved to {output_file}")
#     return True

In [241]:
# --- Helper Functions + Main OCR Function---
def save_entry_to_file(entry_str, file_path="ocr_results.txt"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")
def find_footer_separator_cv2(pil_img):
    # 1. Convert to OpenCV grayscale
    img = np.array(pil_img.convert('L'))
    h, w = img.shape
    
    # 2. Focus on the bottom 70% and binarize
    # We use a higher threshold (220) to catch very faint/thin lines
    start_y = int(h * 0.30)
    roi = img[start_y:, :]
    _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Create a horizontal "needle" kernel
    # This kernel is 1 pixel tall but 100 pixels wide.
    # It acts like a comb that only lets horizontal structures through.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    
    # 4. Apply Morphology (Erosion followed by Dilation)
    # This deletes everything that isn't at least 100 pixels wide horizontally.
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
    # 5. Find the Y-coordinate of the remaining pixels
    # We look for the row with the most white pixels in the 'detected_lines' mask
    row_sums = np.sum(detected_lines, axis=1)
    
    if np.max(row_sums) > 0:
        # Find the index of the row with the maximum "line-ness"
        line_relative_y = np.argmax(row_sums)
        global_y = start_y + line_relative_y
        
        return global_y - 10
        
    return h

def find_dynamic_split_point(pil_img):
    # to divide a page into 2 by identifying the divider between the columns.
    
    # 1. Convert to grayscale and binarize
    bw_arr = np.array(pil_img.convert('L'))
    # 0 = black (text), 1 = white (background)
    bw_arr = (bw_arr > 128).astype(np.uint8) 
    
    h, w = bw_arr.shape
    
    # 2. Focus on the middle 20% of the page width
    # This ensures we don't accidentally split on a margin
    search_start = int(w * 0.35)
    search_end = int(w * 0.65)
    mid_zone = bw_arr[:, search_start:search_end]
    
    # 3. Vertical Projection: Sum the white pixels in each column
    # The gutter will be the column with the MOST white pixels
    column_white_sums = np.sum(mid_zone, axis=0)
    
    # Find the index of the column that is "whitest"
    best_gutter_relative = np.argmax(column_white_sums)
    
    # 4. Return the absolute X coordinate
    dynamic_mid = search_start + best_gutter_relative
    
    # Debug: if the "whitest" column isn't significantly white, fallback to w/2
    if column_white_sums[best_gutter_relative] < (h * 0.95):
        return int(w / 2)
        
    return dynamic_mid

def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn",
                            ocr_config='--oem 3 --psm 3',
                            output_file="ocr_results.txt",
                            export_pics=False, limit_max_p=None):

    max_pages = pdfinfo_from_path(pdf_path)["Pages"]

    # helper: write buffered entry
    def write_entry(entry, marker=None):
        if not entry: return
        s = " ".join(entry).strip()
        if s.endswith(('-', '–', '—')):
            s = s[:-1].strip()
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"{s}\n{marker}\n\n" if marker else f"{s}\n\n")

    open(output_file, "w", encoding="utf-8").close() # create fresh file

    current_entry, held_marker = [], None

    for page_num in tqdm(range(1, max_pages + 1), desc="Processing Dictionary", unit="page"):
        try:
            images = convert_from_path(pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num)
            if not images: continue
            img = images[0]

            # crop footer + split columns
            footer_y = find_footer_separator_cv2(img)
            if footer_y < img.size[1]:
                img = img.crop((0, 0, img.size[0], footer_y))

            w, h = img.size
            mid = find_dynamic_split_point(img)
            columns = [
                img.crop((w * 0.05, h * 0.05, mid * 1.02, h)),
                img.crop((mid, h * 0.05, w * 0.95, h))
            ]

            for i, col_img in enumerate(columns):

                if export_pics:
                    col_img.save(f'p{page_num}_{i+1}.png')

                data = pytesseract.image_to_data(
                    col_img, lang=languages, config=ocr_config, output_type=Output.DICT)
                col_img.close()

                # reconstruct lines from OCR words
                lines, curr_text, curr_x = [], [], None
                for j in range(len(data['text'])):
                    if int(data['conf'][j]) > 0:
                        if int(data['word_num'][j]) == 1:
                            if curr_text:
                                lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                            curr_x, curr_text = data['left'][j], [data['text'][j]]
                        else:
                            curr_text.append(data['text'][j])
                if curr_text:
                    lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                if not lines: continue

                # extract guide word
                first = lines[0]['text'].strip()
                m = re.match(r'^([A-Z]{3})(?:\s+|$)', first)
                guide = m.group(1) if m else ""
                if guide:
                    lines[0]['text'] = first[len(guide):].strip()
                if not lines[0]['text']:
                    lines.pop(0)
                if not lines: continue

                marker = f"COLUMN-START-P{page_num}-C{i+1}"
                marker = f"{marker}-{guide}" if guide else marker

                median_x = np.median([l['x'] for l in lines])
                first_line = lines[0]
                tokens = first_line['text'].split()

                is_new = (
                    first_line['x'] > (median_x + 40) and
                    tokens and tokens[0].endswith(',')
                )

                # hyphen carry across columns
                hyphen_carry = current_entry and current_entry[-1].strip().endswith(('-', '–', '—'))

                if hyphen_carry:
                    current_entry[-1] = current_entry[-1].strip()[:-1] + first_line['text'].strip()
                    held_marker, start_idx = marker, 1

                elif is_new:
                    write_entry(current_entry)
                    current_entry = [first_line['text'].strip()]
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(f"{marker}\n\n")
                    held_marker, start_idx = None, 1

                else:
                    current_entry.append(first_line['text'].strip())
                    held_marker, start_idx = marker, 1

                # process remaining lines
                for line in lines[start_idx:]:
                    txt = line['text'].strip()
                    tokens = txt.split()
                    is_new = (
                        line['x'] > (median_x + 40) and
                        tokens and tokens[0].endswith(',')
                    )

                    if is_new:
                        write_entry(current_entry, held_marker)
                        current_entry, held_marker = [txt], None
                    else:
                        if current_entry and current_entry[-1].strip().endswith(('-', '–', '—')):
                            current_entry[-1] = current_entry[-1].strip()[:-1] + txt
                        else:
                            current_entry.append(txt)

            img.close()
            del images, img
            gc.collect()

        except Exception as e:
            print(f"Skipping page {page_num} due to error: {e}")
            continue

        if limit_max_p and page_num == limit_max_p:
            break

    write_entry(current_entry, held_marker)
    return f"Success! Results appended to {output_file}"

In [247]:
# filename = "input_dict_files/rundi_sample.pdf" # sample with 3 pages
filename = "input_dict_files/rundi_working_file.pdf" # main

start_time = time.perf_counter()
extracted_content = ocr_pdf_lineseg_pipeline(
    
    filename, 
                                     
    image_dpi=600, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = f"--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0",

    output_file = "rundi_extracted_full.txt",

    export_pics = False, # debug

    limit_max_p = None # debug
)

end_time = time.perf_counter() 

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')
extracted_content

Processing Dictionary:   0%|          | 0/588 [00:00<?, ?page/s]

Skipping page 306 due to error: '<' not supported between instances of 'int' and 'NoneType'
Skipping page 524 due to error: '<' not supported between instances of 'int' and 'NoneType'
Time taken to OCR: 80.92 min


'Success! Results appended to rundi_extracted_full.txt'

# Text Cleaning

In [178]:
print(sorted(__import__('collections').Counter(open('rundi_extracted_cleaned_p30.txt', encoding='utf-8').read()).items(), key=lambda x: x[1]))


[('Ê', 1), ('ş', 1), ('Œ', 1), ('ť', 1), ('Ū', 1), ('Ì', 1), ('ż', 1), ('Ó', 1), ('Â', 1), ('ž', 2), ('İ', 2), ('ğ', 2), ('Ī', 3), ('ď', 3), ('É', 3), ('À', 4), ('X', 5), ('ć', 6), ('Z', 6), ('ň', 8), ('Í', 9), ('ì', 9), ('ŭ', 10), ('Q', 10), ('—', 10), ('œ', 11), ('Î', 11), ('æ', 12), ('J', 17), ('«', 18), ('»', 18), ('ò', 21), ('ï', 25), ('ě', 25), ('’', 30), ('W', 33), ('ĉ', 33), ('ç', 34), ('Y', 35), ('?', 38), ('ë', 42), ('D', 43), ('ė', 45), ('H', 63), (':', 65), ('L', 65), ('ē', 65), ('O', 80), ('V', 81), ('G', 82), ('ō', 84), ('!', 86), ('M', 89), ('ù', 90), ('T', 92), ('C', 99), ('ô', 102), ('F', 106), ('í', 108), ('R', 115), ('ū', 123), ('ī', 126), ('ú', 140), ('N', 143), ('B', 146), ('E', 147), ('I', 148), ('û', 151), (';', 159), ('ó', 174), ('î', 175), ('U', 180), ('A', 203), ('9', 249), ('6', 251), ('P', 253), ('8', 254), ('0', 263), ('è', 274), ('5', 274), ('7', 280), ('K', 283), ('x', 292), ('ê', 350), ('ă', 376), ('4', 384), ('2', 405), ('à', 427), ('3', 432), ('ā', 442

In [252]:
# ! IMP --- Create a string set of allowed characters
unique_chars = set(open("rundi_extracted_cleaned_full.txt", "r", encoding="utf-8").read())

# Filter out non-printable/control chars and join with their uppercase forms
whitelist = "".join(sorted({c for c in unique_chars if c.strip()} | {c.upper() for c in unique_chars if c.strip()}))
whitelist+=' '

print(f"{whitelist}")

# with open('rundi_whitelist_char_raw.txt', 'w', encoding='utf-8') as file:
#     file.write(whitelist)

!'(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|«»ÀÁÂÆÇÈÉÊËÌÍÎÏÒÓÔÙÚÛÝàáâæçèéêëìíîïòóôùúûýĀāĂăĆćĈĉĎďĒēĖėĚěĞğĠġĢģĤĥĪīİĴĵŃńŇňŌōŒœŚśŜŝŞşŢŤťŪūŬŭŹźŻżŽž—’ 


In [35]:
import re
import unicodedata

# 0. Load the whitelist once
try:
    with open('rundi_whitelist_char_baseline.txt', 'r', encoding='utf-8') as f:
        whitelist_raw = f.read()
        # Include common structural characters just in case they aren't in your text file
        SAFE_CHARS = set(unicodedata.normalize('NFD', whitelist_raw)) | set(" \n\r\t().,:-")
except FileNotFoundError:
    SAFE_CHARS = set()
    print("Warning: rundi_whitelist_char_baseline.txt not found.")

def clean_rundi_dict(text):
    
    # 1. Global character-level swaps (Pre-decomposition for clarity)
    base_map = {
        # Diacritic Swaps
        'å': 'â', 'ä': 'a', 'ů': 'û', 'ö': 'ō', 'ẹ': 'e', 'ř': 'i','ü':'u','ņ':'n','ċ':'c','ķ':'k','ĝ':'g',
        'ľ':'l', 'š':'','č':'c',
        
        # Stroke/Special Character Base-Mapping
        'đ': 'd', 'ļ': 'l', 'ł': 'l', 'ħ': 'h', 'ø': 'o', 'ŧ': 't',
    }
    
    # 2. Expand to UPPERCASE
    cleanup_map = {
        **base_map, 
        **{k.upper(): v.upper() for k, v in base_map.items()}
    }
    
    # run cleanup on characters
    for search, replace in cleanup_map.items():
        text = text.replace(search, replace)

    
    # 2. Bracket normalization
    text = re.sub(r'[\[\{]', '(', text)
    text = re.sub(r'[\]\}]', ')', text)
    text = re.sub(r'\(+', '(', text)
    text = re.sub(r'\)+', ')', text)

    # 3. Decompose characters
    nfd_text = unicodedata.normalize('NFD', text)
    
    # 4. Define the Diacritic Mapping (Combining Marks)
    DIACRITIC_MAP = {
        '\u030a': '\u0302', # Ring -> Circumflex
        '\u0328': '',       # Ogonek -> Remove
        '\u030b': '\u0301', # Double Acute -> Acute
    }

    # 5. Filter/Map the decomposed string
    cleaned_chars = []
    for char in nfd_text:
        # Check if it is a diacritic (Mark, Nonspacing)
        is_diacritic = unicodedata.category(char) == 'Mn'

        if char in SAFE_CHARS:
            cleaned_chars.append(char)
        elif is_diacritic:
            if char in DIACRITIC_MAP:
                mapped_mark = DIACRITIC_MAP[char]
                if mapped_mark: # Only append if the map doesn't lead to removal
                    cleaned_chars.append(mapped_mark)
            else:
                # Rule: If diacritic not in SAFE or MAP, delete it (do nothing)
                pass
        else:
            # Rule: If base character is not in SAFE_CHARS, skip it
            # This keeps your text strictly within your known character set
            pass
    
    cleaned_nfd = "".join(cleaned_chars)

    # 6. Recompose back to standard characters (NFC)
    return unicodedata.normalize('NFC', cleaned_nfd)

In [36]:
rundi_auto_cleaned = clean_rundi_dict(open("rundi_extracted_full.txt").read())

with open('rundi_extracted_cleaned_full.txt', 'w', encoding='utf-8') as f:
    f.write(rundi_auto_cleaned)

In [38]:
import re

INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_FILE = "rundi_extracted_cleaned_full.txt"

def normalize_column_spacing(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # This replaces the entire "whitespace bridge" with exactly two newlines.
    fixed_content = re.sub(r'\s+(?=COLUMN-START)', r'\n\n', content)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(fixed_content)
    
    print(f"Cleanup complete! All COLUMN-START markers now have exactly two newlines.")

normalize_column_spacing(INPUT_FILE, OUTPUT_FILE)

Cleanup complete! All COLUMN-START markers now have exactly two newlines.


In [92]:
import re
import difflib
from collections import Counter

def extract_target_features(text):
    # common + OCR-misread diacritic characters in our code:
    char_pattern_raw = r"[!'(),-./0123456789:;?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[abcdefghijklmnopqrstuvwxyz{|}«»ÀÁÂÃÄÅÆÇÈÉÊËÍÎÏÒÓÔÖÙÚÛÜàáâãäåæçèéêëíîïòóôöùúûüĀāĂăĄąĆćĈĉĊċČčĐđĒēĖėĚěĦħĪīĶķĻļĽľŁłŅņŌōŒœŘřŠšŪūŬŭŮůŲųẸẹỌọ’„ ]+"

    # Combine patterns
    combined_regex = f"({char_pattern_raw})"
    matches = re.findall(combined_regex, text)
    return matches

def get_feature_diffs(raw_file, clean_file):
    # 1. Read files
    with open(raw_file, 'r', encoding='utf-8') as f:
        raw_tokens = f.read().split()
    with open(clean_file, 'r', encoding='utf-8') as f:
        clean_tokens = f.read().split()

    # 2. Match words (Tokens) to find where changes happened
    matcher = difflib.SequenceMatcher(None, raw_tokens, clean_tokens)
    edit_registry = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        # 'replace' means a word changed; 'delete'/'insert' handles dropped/added words
        if tag in ('replace', 'delete', 'insert'):
            
            # Combine the chunks being compared (handling unequal list lengths)
            r_chunk = "".join(raw_tokens[i1:i2])
            c_chunk = "".join(clean_tokens[j1:j2])

            # 3. Perform Character-Level Diff on the changed chunk
            char_matcher = difflib.SequenceMatcher(None, r_chunk, c_chunk)
            for c_tag, ci1, ci2, cj1, cj2 in char_matcher.get_opcodes():
                if c_tag != 'equal':
                    # Extract the specific character(s) that changed
                    from_chars = r_chunk[ci1:ci2]
                    to_chars = c_chunk[cj1:cj2]
                    
                    # Optional: Filter here if you only want to track vowel/bracket changes
                    if from_chars or to_chars:
                        edit_registry.append(f"{from_chars} -> {to_chars}")

    return Counter(edit_registry).most_common()

    

In [250]:
# AUTO CLEAN FUNCTION RESULT - DIFF
get_feature_diffs("rundi_extracted_full.txt", "rundi_extracted_cleaned_full.txt")

[('å -> â', 2092),
 ('ů -> û', 1740),
 ('} -> ', 1182),
 ('ö -> ō', 1035),
 ('ļ -> l', 742),
 ('} -> )', 716),
 ('ü -> u', 520),
 ('ä -> a', 460),
 ('č -> c', 459),
 ('ð -> ', 444),
 ('{ -> (', 408),
 ('/ -> ', 385),
 ('] -> )', 352),
 ('õ -> o', 315),
 ('ą -> a', 307),
 ('[ -> (', 295),
 ('ł -> l', 288),
 ('đ -> d', 241),
 ('ņ -> n', 220),
 ('ľ -> l', 151),
 ('+ -> ', 137),
 ('ẹ -> e', 124),
 ('ř -> i', 104),
 ('“ -> ', 85),
 ('„ -> ', 78),
 ('ħ -> h', 77),
 ('ã -> a', 74),
 ('` -> ', 68),
 ('ķ -> k', 67),
 ('ő -> ó', 60),
 ('į -> i', 60),
 ('ọ -> o', 56),
 ('= -> ', 55),
 ('ĝ -> g', 52),
 ('^ -> ', 52),
 ('š -> ', 50),
 ('_ -> ', 50),
 ('ı -> ', 50),
 ('ų -> u', 47),
 ('Į -> I', 46),
 ('& -> ', 42),
 ('ę -> e', 42),
 ('ą -> ', 37),
 (' -> a', 37),
 ('~ -> ', 35),
 ('čð -> c', 35),
 ('þ -> ', 26),
 ('ű -> ú', 26),
 ('‘ -> ', 26),
 ('@ -> ', 25),
 ('ļ] -> l)', 25),
 ('}) -> ', 24),
 ('Ķ -> K', 24),
 ('{ -> ', 23),
 ('ċ -> c', 19),
 ('\\ -> ', 18),
 ('° -> ', 18),
 ('ðč -> c', 17),
 ('ð

In [251]:
raw_charset = set(open("rundi_extracted_cleaned_full.txt", "r", encoding="utf-8").read())
clean_charset = set(open("rundi_whitelist_char_baseline.txt", "r", encoding="utf-8").read())

" ".join(char for char in raw_charset if char not in clean_charset)
# lists all the weird OCR characters after cleaning

'Ž İ \n ğ ť Ş Ţ ď ž Ď ý ĵ ň Ġ ġ Ğ ż ŝ Ì ş ś ź ģ ń ĥ'

# Ollama Prompting

In [267]:
# MLX Implementation of Ollama - Unified Memory

In [22]:
# !export OLLAMA_MLX=1
# !ollama serve

]11;?\Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [18]:
import ollama
import pandas as pd
import json
import time
from tqdm.notebook import tqdm
import os


def fast_batch_process():
    # 1. Load and Chunk
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        all_entries = [e.strip() for e in f.read().split('\n\n') if len(e.strip()) > 3 and not e.strip().upper().startswith("COLUMN-START")]

    # final_results = []
    # create fresh csv file
    with open(OUTPUT_CSV, 'w') as f:
        f.write('')
        
    # 2. Process in Batches
    # Note: If still missing entries, consider BATCH_SIZE = 2 or 3
    for i in tqdm(range(0, len(all_entries), BATCH_SIZE), desc="Processing Batches", unit='batch'):
        batch = all_entries[i : i + BATCH_SIZE]
        
        # Use XML tags to make the boundaries "harder" for the model
        batch_str = "\n".join([f"<entry id='{idx}'>{text}</entry>" for idx, text in enumerate(batch)])
        
        for attempt in range(2):
            try:
                response = ollama.chat(
                    model=MODEL,
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user', 'content': f"Extract every entry within the tags into a JSON array labeled 'entries':\n{batch_str}"}
                    ],
                    format='json',
                    options={
                        'temperature': 0,
                        'num_ctx': 4096,   # Context window size
                        'num_predict': 2048, # Max output length
                        'top_k': 1,
                        'top_p': 0.01
                    },
                    think=False
                )
                
                content = response['message']['content']
                batch_data = json.loads(content)
                
                # Robust key extraction
                if isinstance(batch_data, dict) and 'entries' in batch_data:
                    items = batch_data['entries']
                elif isinstance(batch_data, list):
                    items = batch_data
                else:
                    items = [batch_data]

                # final_results.extend(items)
                break 
                
            except Exception as e:
                if attempt == 1:
                    print(f"Error:{e}\nSkipping batch at index {i} after failures.")
                else:
                    time.sleep(2)
                    continue

        # 3. Save Every Even Batch (Append Mode)
        # Check if file exists to determine if we need a header
        # Save to CSV
        pd.DataFrame(items).to_csv(
            OUTPUT_CSV, 
            mode='a',             # Append mode
            index=False, 
            header= i==0, # Only write header if file is new
            encoding='utf-8'
        )
        
        # CRITICAL: Clear the list so we don't save duplicates in the next save
        # final_results = []
    # Final Save
    # df = pd.DataFrame(final_results)
    # df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nExtraction complete. Total entries: {len(df)}")

In [21]:

# Run it
start_time = time.perf_counter()

# --- CONFIGURATION ---
# MODEL = "llama3.2:3b"
MODEL = "gemma4:e2b"
# MODEL = "gemma2:2b"
BATCH_SIZE = 4

INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_CSV = f"rundi_llm_tagged_{MODEL.split(":")[0]}_B{BATCH_SIZE}.csv"


SYSTEM_PROMPT="""
Return a JSON object with a single key "entries" containing an array of objects.
For every <entry> provided, extract:
- WORD: The Rundi headword.
- PLURAL: The prefix or suffix containing a hyphen.
- NOUN_CLASS: The numeric class pair (e.g., "3|4").
- DESCRIPTION: The first French definition.
- POS: The part-of-speech abbreviation.

If a field is missing, use null. Do not skip any entries.
"""

fast_batch_process()

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')


Processing Batches:   0%|          | 0/4854 [00:00<?, ?batch/s]

KeyboardInterrupt: 

# Osaurus.ai

In [18]:
from openai import OpenAI
import pandas as pd
import json
import time
import os
from tqdm.notebook import tqdm

# --- CONFIG ---
MODEL = "gemma-4-e2b-it-4bit"
INPUT_FILE = "rundi_extracted_cleaned_full.txt"
OUTPUT_CSV = "rundi_llm_tagged_gemma4_osaurus.csv"
BATCH_SIZE = 4

EXPECTED_KEYS = ["WORD", "PLURAL", "NOUN_CLASS", "DESCRIPTION", "POS"]

client = OpenAI(base_url="http://127.0.0.1:1337/v1", api_key="osaurus")

SYSTEM_PROMPT = """
Return a JSON object with a single key "entries" containing an array of objects.

For EVERY <entry>:
- WORD: The Rundi headword
- PLURAL: The prefix or suffix
- NOUN_CLASS: The numeric class pair (e.g., "3|4")
- DESCRIPTION: The first French definition sentence
- POS: The part-of-speech abbreviation

RULES:
- Do not skip entries
- If missing → null
- Output MUST be valid JSON
- No markdown
- No text outside JSON
"""

# --- CLEAN START ---
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)


# --- LOAD DATA ---
def load_entries():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        return [
            e.strip()
            for e in f.read().split("\n\n")
            if len(e.strip()) > 10
            and not e.strip().upper().startswith("COLUMN-START")
        ]


# --- NORMALIZE ---
def normalize(entry):
    return {
        "WORD": entry.get("WORD"),
        "PLURAL": entry.get("PLURAL"),
        "NOUN_CLASS": entry.get("NOUN_CLASS"),
        "DESCRIPTION": entry.get("DESCRIPTION"),
        "POS": entry.get("POS"),
    }


# --- MAIN PIPELINE ---
def extract_tags_osaurus():
    all_entries = load_entries()

    final_results = []
    batch_count = 0

    for i in tqdm(range(0, len(all_entries), BATCH_SIZE), desc="Processing", unit="batch"):
        batch = all_entries[i : i + BATCH_SIZE]
        batch_count += 1

        # 🔥 EXACTLY like Ollama: ID-tagged XML
        batch_str = "\n".join([
            f"<entry id='{idx}'>{text}</entry>"
            for idx, text in enumerate(batch)
        ])

        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {
                            "role": "user",
                            "content": f"Extract EVERY entry into JSON:\n{batch_str}"
                        }
                    ],
                    temperature=0,
                    top_p=0.01,   # 🔥 mimic ollama
                )

                content = response.choices[0].message.content.strip()

                # --- STRICT JSON PARSE ---
                batch_data = json.loads(content)

                # --- SAME LOGIC AS OLLAMA ---
                if isinstance(batch_data, dict) and "entries" in batch_data:
                    items = batch_data["entries"]
                elif isinstance(batch_data, list):
                    items = batch_data
                else:
                    items = [batch_data]

                # Normalize + enforce schema
                for item in items:
                    final_results.append(normalize(item))

                break  # success

            except Exception as e:
                if attempt == 2:
                    print(f"Error: {e}\n❌ Skipping batch {i} after 3 failures")
                else:
                    time.sleep(2)

        # --- SAVE EVERY EVEN BATCH ---
        if batch_count % 2 == 0:
            pd.DataFrame(final_results, columns=EXPECTED_KEYS).to_csv(
                OUTPUT_CSV, index=False
            )

    # --- FINAL SAVE ---
    df = pd.DataFrame(final_results, columns=EXPECTED_KEYS)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n✅ Done. Total entries: {len(df)}")

In [19]:
start_time = time.perf_counter()

extract_tags_osaurus()

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')

Processing:   0%|          | 0/4853 [00:00<?, ?batch/s]

Error: Expecting value: line 1 column 1 (char 0)
❌ Skipping batch 0 after 3 failures


KeyboardInterrupt: 

# Manual Tagging

In [44]:
import re
import pandas as pd

def extract_with_ocr_fixes(text_block):
    # Updated NC Pattern for messy OCR like '9lal2a' or '9i|10i'
    # Logic: Digit + any non-comma chars (greedy) + Digit + any non-comma chars
    nc_pattern = r"\b\d+[^,]*?\d+[^,]*"
    
    # Fallback for single noun classes (e.g., '11')
    single_nc_pattern = r"\b\d{1,2}[a-z]?\b"
    
    # Added common dictionary tags to the exclusion list
    pos_tags = ["v.", "n.", "adj.", "v", "n", "V", "N", "adv.", "interj.", "inv.", "(inv.)"]
    extracted_data = []
    
    # Split by double newline and filter noise
    entries = [e.strip() for e in text_block.split('\n\n') if len(e.strip()) > 10]

    for entry in entries:
        if entry.startswith("COLUMN-START"): continue
        
        # Clean up whitespace but preserve commas for splitting
        full_entry = " ".join(entry.split())
        parts = [p.strip() for p in full_entry.split(',')]
        if len(parts) < 2: continue

        word = parts[0]
        plural = None
        noun_class = None
        
        # --- POSITIONAL ANALYSIS (Segments 2 and 3) ---
        for i in range(1, 3):
            if i >= len(parts): break
            p = parts[i]
            
            # 1. Check for Complex Noun Class (e.g., 9lal2a)
            # We ensure it's not a known POS tag before matching
            if p.lower() not in pos_tags:
                nc_match = re.search(nc_pattern, p)
                
                # 2. Check for Single Noun Class (e.g., 11) if complex failed
                if not nc_match and re.fullmatch(single_nc_pattern, p):
                    nc_match = re.search(single_nc_pattern, p)
                
                if nc_match:
                    noun_class = nc_match.group(0).strip()
                    
                    # POSITIONAL RULE: If NC is in the 3rd slot, 2nd slot MUST be Plural
                    if i == 2:
                        plural = parts[1]
                    break

        # --- PLURAL FALLBACK ---
        # Catch hyphenated plurals if the positional rule didn't trigger
        if not plural and len(parts) > 1:
            p1 = parts[1]
            if '-' in p1:
                plural = p1

        # --- DESCRIPTION EXTRACTION ---
        description = None
        # Skip the word and metadata fields already found
        start_search = 1
        if plural: start_search += 1
        if noun_class: start_search += 1
        
        for i in range(start_search, len(parts)):
            p = parts[i]
            # French sentences usually start with a Capital or Digit
            if p and (p[0].isupper() or p[0].isdigit()) and p != noun_class:
                remaining = ", ".join(parts[i:])
                # Capture until the first period
                desc_match = re.search(r"([^.!?]+[.!?])", remaining)
                description = desc_match.group(1) if desc_match else p
                break

        # --- POS EXTRACTION ---
        # Look for the tag near the end of the entry
        pos_match = re.search(r"\b(" + "|".join([re.escape(t) for t in pos_tags]) + r")\b\.?$", full_entry)
        pos = pos_match.group(1) if pos_match else None

        extracted_data.append({
            "WORD": word,
            "PLURAL": plural,
            "NOUN_CLASS": noun_class,
            "DESCRIPTION": description,
            "POS": pos,
            "FULL_ENTRY": full_entry
        })
            
    return pd.DataFrame(extracted_data)

In [45]:

with open("rundi_extracted_cleaned_full.txt", "r") as f:
    text = f.read()
df = extract_improved_rundi(text)
df.to_csv("rundi_tagged_regex.csv", index=False)